In [18]:
""" 
Writing Out Melbourne Solar Lull Events
=====================================================
Description: This notebook will open Solar data files, define solar drought for melbourne, write out a CSV file

Version History: 17/06/26, written by A.D.

Notes:
"""

' \nWriting Out Melbourne Solar Lull Events\n=====================================================\nDescription: This notebook will open Solar data files, define solar drought for melbourne, write out a CSV file\n\nVersion History: 17/06/26, written by A.D.\n\nNotes:\n'

In [19]:
import xarray as xr
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [20]:
path = '/g/data/gb02/cd3022/solar-pv/GCCSA/2GMEL_ideal-actual_timeseries.nc'
melb = xr.open_dataset(path)
melb

<xarray.Dataset> Size: 9MB
Dimensions:  (time: 389959)
Coordinates:
  * time     (time) datetime64[ns] 3MB 2015-07-31T18:30:00 ... 2025-12-31T11:...
Data variables:
    actual   (time) float64 3MB ...
    ideal    (time) float64 3MB ...

In [21]:
# Change times to AEST (timezone used by AEMO)
time_utc = pd.to_datetime(melb.time.values)
time_aest = time_utc.tz_localize("UTC").tz_convert("Australia/Brisbane")
time_aest_naive = time_aest.tz_convert("Australia/Brisbane").tz_localize(None)
melb = melb.assign_coords(time=("time", time_aest_naive))

# The satellite undergoes maintenance and skips a timestep once a day, so remove this time
melb = melb.where(melb['time'].dt.strftime('%H:%M') != '12:40', drop=True)

In [22]:
#calcualte csi 
melb['csi'] = melb['actual'] / melb['ideal'] 

# Where CSI is great than 1, flatten values to 1.
# Because CSI >= 1 indicates a clear sky, and large values can distort the norm
melb['csi'] = xr.where(melb.csi > 1, 1, melb.csi)

In [23]:
#define solar lull 
# any day where a region's daily mean CSI is less than 0.5
daily = melb.resample(time='1D').mean()
csi_lull = xr.where(daily.csi < 0.5, 1, 0)

In [24]:
#Save as a csv

# da is your DataArray
da = csi_lull

# Create a new dataset with a new variable name
ds_new = xr.Dataset({"lull": da})
ds_new

#Save as a CSV, first convert xr to pd
df = ds_new.to_dataframe()

# Save it 
df.to_csv("/scratch/nf33/ad1803/GC26_Solar_Lull/Work/Data/Lulls/Melb_lulls.csv")